# ML-06 — Signal Audit: Do the Flags Hold?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dv-06/flyrank-ml_1/blob/main/work/notebooks/w04_signal_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

!git clone https://github.com/dv-06/flyrank-ml_1.git
%cd flyrank-ml_1
import pandas as pd

# Load the anonymized dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Key signals used by the content-refresh rule
features = [
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]

print("Dataset shape:", df.shape)

print("\nKey feature statistics:")
print(df[features].describe().round(2))

print("\nMedian values:")
print(df[features].median().round(2))

print("\nTrend direction distribution:")
print(df["trend_direction"].value_counts(dropna=False))



Cloning into 'flyrank-ml_1'...
remote: Enumerating objects: 236, done.
remote: Counting objects: 100% (236/236), done.
remote: Compressing objects: 100% (190/190), done.
remote: Total 236 (delta 114), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (236/236), 1.97 MiB | 15.60 MiB/s, done.
Resolving deltas: 100% (114/114), done.
/content/flyrank-ml_1/flyrank-ml_1/flyrank-ml_1/flyrank-ml_1/flyrank-ml_1
Dataset shape: (30000, 44)

Key feature statistics:
       impressions_90d       ctr  avg_position  content_age_days
count         30000.00  30000.00      30000.00          30000.00
mean           5200.37      0.51         16.34            256.17
std           16838.02      3.28         15.22            132.71
min               1.00      0.00          0.00             90.00
25%              81.00      0.00          6.20            132.00
50%             731.00      0.07         10.80            236.00
75%            3615.25      0.29         22.30            333.00
max

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Test whether the three rule signals are directionally supported
# by the observed trend_direction field.

def find_declining_category(values):
    """
    Find the trend_direction category representing declining/downward performance.
    """
    categories = values.dropna().astype(str).str.lower().unique()

    for category in categories:
        if any(word in category for word in [
            "declin", "down", "fall", "drop", "negative", "wors"
        ]):
            return category

    return None


declining_category = find_declining_category(df["trend_direction"])

print("Detected declining category:", declining_category)

if declining_category is None:
    print("\nCould not automatically identify a declining category.")
    print("Available categories:")
    print(df["trend_direction"].dropna().unique())
else:
    df["_is_declining"] = (
        df["trend_direction"]
        .astype(str)
        .str.lower()
        == declining_category
    )

    tests = [
        ("High impressions", "impressions_90d", "high"),
        ("Low CTR", "ctr", "low"),
        ("Poor average position", "avg_position", "high")
    ]

    print("\nSIGNAL TEST RESULTS")
    print("=" * 70)

    for name, feature, direction in tests:

        q25 = df[feature].quantile(0.25)
        q75 = df[feature].quantile(0.75)

        if direction == "high":
            signal_group = df[df[feature] >= q75]
            comparison_group = df[df[feature] <= q25]
        else:
            signal_group = df[df[feature] <= q25]
            comparison_group = df[df[feature] >= q75]

        signal_rate = signal_group["_is_declining"].mean()
        comparison_rate = comparison_group["_is_declining"].mean()

        difference = signal_rate - comparison_rate

        if difference >= 0.05:
            verdict = "CONFIRMED"
        elif difference <= -0.05:
            verdict = "OPPOSITE"
        else:
            verdict = "MIXED"

        print(f"\n{name}")
        print(f"Feature: {feature}")
        print(f"Signal group size: {len(signal_group)}")
        print(f"Comparison group size: {len(comparison_group)}")
        print(f"Declining rate in signal group: {signal_rate:.3f}")
        print(f"Declining rate in comparison group: {comparison_rate:.3f}")
        print(f"Difference: {difference:+.3f}")
        print(f"Verdict: {verdict}")

Detected declining category: down

SIGNAL TEST RESULTS

High impressions
Feature: impressions_90d
Signal group size: 7500
Comparison group size: 7503
Declining rate in signal group: 0.562
Declining rate in comparison group: 0.376
Difference: +0.186
Verdict: CONFIRMED

Low CTR
Feature: ctr
Signal group size: 13212
Comparison group size: 7504
Declining rate in signal group: 0.497
Declining rate in comparison group: 0.520
Difference: -0.023
Verdict: MIXED

Poor average position
Feature: avg_position
Signal group size: 7508
Comparison group size: 7543
Declining rate in signal group: 0.517
Declining rate in comparison group: 0.461
Difference: +0.055
Verdict: CONFIRMED


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Flag-linked audit:
# Does the "Low CTR" rule correspond to a higher observed
# rate of declining trend_direction?

if "_is_declining" not in df.columns:
    declining_category = find_declining_category(df["trend_direction"])

    if declining_category is not None:
        df["_is_declining"] = (
            df["trend_direction"]
            .astype(str)
            .str.lower()
            == declining_category
        )

if "_is_declining" in df.columns:

    q25 = df["ctr"].quantile(0.25)
    q75 = df["ctr"].quantile(0.75)

    low_ctr = df[df["ctr"] <= q25]
    high_ctr = df[df["ctr"] >= q75]

    low_ctr_rate = low_ctr["_is_declining"].mean()
    high_ctr_rate = high_ctr["_is_declining"].mean()

    difference = low_ctr_rate - high_ctr_rate

    if difference >= 0.05:
        verdict = "CONFIRMED"
    elif difference <= -0.05:
        verdict = "OPPOSITE"
    else:
        verdict = "MIXED"

    print("FLAG-LINKED TEST: LOW CTR")
    print("=" * 50)
    print(f"Low CTR threshold: <= {q25:.3f}")
    print(f"High CTR threshold: >= {q75:.3f}")

    print(f"\nLow-CTR pages: {len(low_ctr)}")
    print(f"High-CTR pages: {len(high_ctr)}")

    print(f"\nDeclining rate among low-CTR pages: {low_ctr_rate:.3f}")
    print(f"Declining rate among high-CTR pages: {high_ctr_rate:.3f}")

    print(f"\nDifference: {difference:+.3f}")
    print(f"Verdict: {verdict}")

else:
    print("Could not identify a declining trend category.")
    print("Available trend_direction values:")
    print(df["trend_direction"].value_counts(dropna=False))

FLAG-LINKED TEST: LOW CTR
Low CTR threshold: <= 0.000
High CTR threshold: >= 0.290

Low-CTR pages: 13212
High-CTR pages: 7504

Declining rate among low-CTR pages: 0.497
Declining rate among high-CTR pages: 0.520

Difference: -0.023
Verdict: MIXED


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Summarize the practical use of the audit.

print("Practical interpretation")
print("=" * 50)

print("The signal tests provide directional evidence about")
print("which page characteristics are associated with declining")
print("performance in the observed dataset.")

print("\nThe signals can be used to prioritize pages for human review.")

print("\nThe results do not establish causation and should not")
print("be treated as proof that refreshing a page will improve")
print("its search performance.")

print("\nDecision: use the signals for editorial prioritization,")
print("with human review before taking action.")


Practical interpretation
The signal tests provide directional evidence about
which page characteristics are associated with declining
performance in the observed dataset.

The signals can be used to prioritize pages for human review.

The results do not establish causation and should not
be treated as proof that refreshing a page will improve
its search performance.

Decision: use the signals for editorial prioritization,
with human review before taking action.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.